In [2]:
# Please install OpenAI SDK first: `pip3 install openai`
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "what is your name"},
    ],
    stream=False,
    reasoning_effort="low",
    extra_body={"thinking": {"type": "disabled"}}
)

print(response.choices[0].message.content)

My name is **DeepSeek**! I'm an AI assistant created by DeepSeek (深度求索). I'm here to help answer your questions, assist with tasks, or just chat. What can I do for you today? 😊


In [3]:
import json
from pprint import pprint

message = response.choices[0].message

# 方式1：转成 dict，去掉 None 字段（推荐）
pprint(message.model_dump(exclude_none=True))

# 方式2：JSON 格式，缩进更好看
print(json.dumps(message.model_dump(exclude_none=True), indent=2, ensure_ascii=False))

{'content': "My name is **DeepSeek**! I'm an AI assistant created by DeepSeek "
            "(深度求索). I'm here to help answer your questions, assist with "
            'tasks, or just chat. What can I do for you today? 😊',
 'role': 'assistant'}
{
  "content": "My name is **DeepSeek**! I'm an AI assistant created by DeepSeek (深度求索). I'm here to help answer your questions, assist with tasks, or just chat. What can I do for you today? 😊",
  "role": "assistant"
}


model_dump(exclude_none=True)来自 **Pydantic**——OpenAI SDK 的返回对象（比如 `response.choices[0].message`）底层都是 Pydantic 的 `BaseModel` 子类。

## `model_dump()` 

把一个 Pydantic 模型对象**递归转换成普通的 Python 字典**。

`response.choices[0].message` 是一个对象，直接打印只能看到对象的 repr，不能用 JSON 序列化，也不能方便地按 key 取值遍历。`model_dump()` 就是官方提供的「对象 → dict」出口：

```python
message = response.choices[0].message
print(message)              # ChatCompletionMessage(content='我是...', refusal=None, ...)
print(message.content)      # 属性访问，一次取一个字段

d = message.model_dump()    # 变成 dict
print(d["content"])         # 按 key 访问
print(json.dumps(d))        # 能直接 JSON 序列化了
```

如果 message 里有嵌套结构（比如 `tool_calls`），`model_dump()` 会一层层全部转成 dict，不是只转最外层。

## `exclude_none=True` 

**转换时把值为 `None` 的字段直接丢掉，不放进结果字典里。**

原因：OpenAI 返回的 message 对象里很多字段你没用到，Pydantic 会把它们填成 `None`（而不是省略）。不做排除的话，dump 出来是这样的：

```python
# 不加 exclude_none
{
    "content": "我是 DeepSeek...",
    "refusal": None,
    "role": "assistant",
    "annotations": None,
    "audio": None,
    "function_call": None,
    "tool_calls": None,
    "reasoning_content": None
}
```

加了之后：

```python
# exclude_none=True
{
    "content": "我是 DeepSeek...",
    "role": "assistant"
}
```

干净很多，尤其当你想把这条 message 塞回 `messages` 列表做多轮对话时——有些 API 对 `null` 字段会报错或行为异常，去掉 `None` 字段更安全。

## 相关的兄弟参数

顺带一提同一个方法还有几个常用参数：

| 参数 | 作用 |
|---|---|
| `exclude_none=True` | 排除值为 `None` 的字段 |
| `exclude_unset=True` | 排除「没被显式赋值」的字段（只保留了默认值的也去掉） |
| `exclude={"audio", "annotations"}` | 按名字精确排除指定字段 |
| `include={"content", "role"}` | 只保留指定字段 |

这里的用法（`model_dump(exclude_none=True)`）是最常见的选择：保留所有有意义的数据，只剔除空值。